# Differentiable neural computer

The DNC (Graves et al., 2016) extends the NTM with three key improvements:

1. **Usage-based allocation**: tracks which memory rows are free, allocates
   the least-used row for writes (no shift-based addressing needed)
2. **Temporal link matrix**: records the order in which memory was written,
   enabling forward/backward traversal of stored sequences
3. **Multi-head reads with mode mixture**: each read head blends content-based,
   forward-link, and backward-link addressing via learned mode weights

**Reference:** Graves et al., "Hybrid computing using a neural network
with dynamic external memory" (Nature, 2016)

**CLI equivalents:** `make example-dnc-copy` and `make example-dnc-recall`


## Architecture

The DNC has an additional type parameter `r` for the number of read heads.
Each read head can independently address memory using different modes. Like
`Ntm`, `Dnc` implements `Nn.Recurrent`: the machine is one recurrent cell
threaded through `recurStep`.


In [ ]:
:t dnc

In [ ]:
:t Dnc

## NTM vs DNC

| Feature | NTM | DNC |
|---------|-----|-----|
| Write addressing | Content + shift | Usage-based allocation |
| Read addressing | Content + shift | Content + temporal links |
| Memory management | None (overwrite) | Free gates + usage tracking |
| Read heads | 1 | R (type parameter) |
| Link matrix | None | O(N^2) temporal ordering |

The DNC is more powerful but slower per step (link matrix is O(N^2)).
Default N=32 (vs NTM's N=128) to keep the link matrix manageable.


## Training

Same copy task and two-phase training as the NTM. The compiled example uses
R=1 read head, N=32 memory rows (reduced from the NTM's 128 because the link
matrix is O(N²)), M=20, H=100, with the same RMSprop configuration, from
`Example/DncCopy.idr`:

```idris
model <- runInitL (dnc {r = R} {n = N} {m = M} {h = H} {i = InputW} {o = OutputW})
```


Synthetic binary-copy data is generated by `copyTaskBinaryBatchVect` in
`packages/idris-ml-examples/src/Generate.idr` — that lives in the examples
package, not the kernel prelude. Run the CLI demo to see training in action:

```bash
make example-dnc-copy DNC_COPY_ARGS="--epochs 20"
```


## DNC memory mechanics

### Allocation
The DNC tracks *usage* of each memory row. When writing, it allocates
the least-used row via argsort + cumprod. Free gates allow the controller
to explicitly release memory rows after reading.

### Temporal links
The link matrix `L[i,j]` records "row i was written immediately after row j".
This enables:
- **Forward read**: follow the write order (L * w_prev)
- **Backward read**: reverse the write order (L^T * w_prev)

### Read modes
Each read head has 3 mode weights (softmax):
- Backward link weighting
- Content-based weighting
- Forward link weighting

The controller learns which mode to use for each head at each timestep.


## Scaling up

For full convergence (the default epoch count is already 50000):
```bash
make example-dnc-copy DNC_COPY_ARGS="--epochs 50000 --lr 0.0001"
make example-dnc-recall DNC_RECALL_ARGS="--epochs 50000 --lr 0.0001"
```

R=1 exercises all DNC mechanisms. R=4 matches the original paper and adds
capacity, but requires more epochs to converge (`R` is a compile-time
constant in `Example/DncCopy.idr`, not a flag).


## PyTorch comparison

The DNC is among the most complex architectures in the library.
The PyTorch implementation requires careful numerical clamping
at 6 points to prevent NaN during multi-timestep forward passes.
idris-ml handles this identically in the C backend.

See `pytorch/torch_ref/scripts/dnc_copy.py` and `dnc_recall.py`
for the full references.


Next: [CNN](cnn.ipynb) — convolutional networks for image classification.
